# Rutinas de apoyo para exportar modelos a TFLite (para despliegue en Smartphone o RaspberryPi) y lenguaje C para ESP32

In [ ]:
import tensorflow as tf
from tensorflow import keras
print("Versión TF", tf.__version__)

## Exportar modelos a formato TFLite

Dependiendo de su arquitectura de hardware destino (Smartphone, RaspberryPi o ESP32) y tamaño requerido para almacenar los modelos en KB, se recomienda usar la alternativa 1 o 2.

La alternativa 2 produce modelos más pequeños pero con pérdida de precisión, y es la única alternativa posible para dispositivos ESP32.

Solamente copie y pegue el código de la alternativa al final de su cuaderno donde entrena el modelo para exportarlo a un archivo con extensión .TFLITE

**Alternativa 1**: Exportar modelo a formato *FLOAT32*

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("modelo.tflite", "wb") as f:
    f.write(tflite_model)

**Alternativa 1**: Exportar modelo a formato *INT8*

Puede usar también este prompt:
> Exporta el modelo a un formato TFLite que sea compatible con Tensorflow Lite Micro empleando cuantización INT8

In [ ]:
import tensorflow as tf
import numpy as np

# 1. Definir un generador de datos representativos utilizando el dataset existente
def representative_data_gen():
    # Usamos un lote pequeño del dataset para la calibración
    for input_value, _ in val_ds.take(20):
        # Convertimos las imágenes a tipo float32
        yield [tf.cast(input_value, tf.float32)]

# 2. Configurar el convertidor de TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Habilitar optimizaciones
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Configurar el generador de datos representativos obligatorio para INT8
converter.representative_dataset = representative_data_gen

# Restringir las operaciones para asegurar compatibilidad total con TFLite Micro
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Configurar entrada y salida del modelo de inferencia en INT8
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# 3. Convertir el modelo
tflite_quant_model = converter.convert()

# 4. Guardar el archivo cuantizado
tflite_model_path = "modelo.tflite"
with open(tflite_model_path, "wb") as f:
    f.write(tflite_quant_model)

print(f"Modelo cuantizado en INT8 exportado exitosamente a: {tflite_model_path}")

## Función de apoyo para crear el archivo de cabera .H requerido para el programa en C sobre la ESP32

In [ ]:
# Script en Python puro para emular el comando xxd y exportar el modelo a C
# Comando original: xxd -i modelo.tflite > modelo.h

def export_tflite_to_c(input_filename, output_filename):
    with open(input_filename, "rb") as f:
        bytes_data = f.read()
    var_name = input_filename.replace(".", "_")
    with open(output_filename, "w") as f:
        f.write(f"// Arreglo de bytes del modelo generado por Python\n")
        f.write(f"const unsigned char {var_name}[] = {{\n    ")
        for i, byte in enumerate(bytes_data):
            f.write(f"0x{byte:02x}")
            if i < len(bytes_data) - 1:
                f.write(", ")
            if (i + 1) % 12 == 0:
                f.write("\n    ")
        f.write("\n};\n\n")
        f.write(f"const unsigned int {var_name}_len = {len(bytes_data)};\n")

    print(f"¡Archivo '{output_filename}' creado con éxito en Python!")

Ejemplo de uso:

In [ ]:
export_tflite_to_c('modelo.tflite', 'clasificador.h')